In [1]:
from ultralytics import YOLO
import torch
from torchvision import transforms
from PIL import Image

# Load YOLO detector
yolo_model = YOLO(r"D:\IITBHU Internship\code\runs\detect\train8\weights\best.pt")  # your trained YOLOv11n

# Load CNN classifier
from torchvision import models
import torch.nn as nn

cnn_model = models.resnet18()
cnn_model.fc = nn.Linear(cnn_model.fc.in_features, 2)  # drone / non_drone
cnn_model.load_state_dict(torch.load("drone_classifier.pth"))
cnn_model.eval()

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
cnn_model.to(device)

# Transform for CNN
transform = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.ToTensor(),
    transforms.Normalize([0.5,0.5,0.5],[0.5,0.5,0.5])
])

# Classes
classes = ["drone","non_drone"]

def multimodel_predict(image_path):
    # Stage 1: YOLO detection
    results = yolo_model(image_path)[0]

    final_detections = []
    for box in results.boxes:
        x1, y1, x2, y2 = map(int, box.xyxy[0].tolist())
        conf = box.conf[0].item()

        # Crop detected region
        img = Image.open(image_path).convert("RGB")
        crop = img.crop((x1,y1,x2,y2))

        # Stage 2: CNN classification
        crop_tensor = transform(crop).unsqueeze(0).to(device)
        with torch.no_grad():
            output = cnn_model(crop_tensor)
            _, pred = torch.max(output, 1)
            label = classes[pred.item()]

        if label == "drone":
            final_detections.append({
                "bbox": (x1,y1,x2,y2),
                "confidence": conf,
                "label": "drone"
            })

    return final_detections

# Example usage
pa = r'D:\IITBHU Internship\code\DroneDatasetCombined\outliers\images\5.JPEG'

detections = multimodel_predict(r"D:\IITBHU Internship\code\DroneDatasetCombined\images\val\0064.jpg")
img = Image.open(pa).convert("RGB")
show_img = img.show()
print("Final detections:", detections)

C:\Users\100ra\AppData\Local\Temp\ipykernel_28048\3992254466.py:15: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  cnn_model.load_state_dict(torch.load("drone_classifier.pth"


image 1/1 D:\IITBHU Internship\code\DroneDatasetCombined\images\val\0064.jpg: 448x768 1 drone, 66.7ms
Speed: 3.8ms preprocess, 66.7ms inference, 11.2ms postprocess per image at shape (1, 3, 448, 768)
Final detections: [{'bbox': (92, 50, 416, 222), 'confidence': 0.8205313682556152, 'label': 'drone'}]


## **Real-time detection**

In [3]:
import cv2
from ultralytics import YOLO
import torch
from torchvision import models, transforms
from PIL import Image
import numpy as np

# -----------------------------
# 1) Load YOLO detector
# -----------------------------
yolo_model = YOLO(r"D:\IITBHU Internship\code\runs\detect\train8\weights\best.pt")

# -----------------------------
# 2) Load CNN classifier
# -----------------------------
cnn_model = models.resnet18()
cnn_model.fc = torch.nn.Linear(cnn_model.fc.in_features, 2)  # drone / non_drone
cnn_model.load_state_dict(torch.load("drone_classifier.pth", map_location="cpu"))
cnn_model.eval()

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
cnn_model.to(device)

# Transform for CNN
transform = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.ToTensor(),
    transforms.Normalize([0.5,0.5,0.5],[0.5,0.5,0.5])
])

classes = ["drone","non_drone"]

# -----------------------------
# 3) Webcam access
# -----------------------------
cap = cv2.VideoCapture(0)

if not cap.isOpened():
    print("Error: Webcam not opening")
    exit()

while True:
    ret, frame = cap.read()
    if not ret:
        print("Frame not found, breaking...")
        break

    # Stage 1: YOLO detection
    results = yolo_model(frame, conf=0.25, device=0, verbose=False)

    # Copy frame for annotation
    annotated_frame = frame.copy()

    # Stage 2: CNN classification for each detection
    for box in results[0].boxes:
        x1, y1, x2, y2 = map(int, box.xyxy[0].tolist())
        conf = box.conf[0].item()

        # Crop region
        crop = frame[y1:y2, x1:x2]
        if crop.size == 0:
            continue

        # Convert to PIL for CNN
        crop_pil = Image.fromarray(cv2.cvtColor(crop, cv2.COLOR_BGR2RGB))
        crop_tensor = transform(crop_pil).unsqueeze(0).to(device)

        with torch.no_grad():
            output = cnn_model(crop_tensor)
            _, pred = torch.max(output, 1)
            label = classes[pred.item()]

        # Only draw if CNN confirms "drone"
        if label == "drone":
            cv2.rectangle(annotated_frame, (x1,y1), (x2,y2), (0,255,0), 2)
            annotated_frame = results[0].plot()  # numpy array (BGR)


    # Show in window
    cv2.imshow("Multimodel Drone Detection", annotated_frame)

    # 'q' to exit
    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

# Cleanup
cap.release()
cv2.destroyAllWindows()

C:\Users\100ra\AppData\Local\Temp\ipykernel_28048\2624111493.py:18: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  cnn_model.load_state_dict(torch.load("drone_classifier.pth"